# Phase 3 - g1-Projected CAPGD on LCLD

Tier C (formula-aware) evaluation per `docs/constraint_evaluation_guidance.md`
section 5. Runs two CAPGD variants side-by-side on the same trained MLP per
seed:

1. **Unconstrained** - stock `capgd_attack` from `attacks/capgd.py` (same
   attack used in every existing LCLD experiment in `registry_clean.csv`).
2. **g1-projected** - identical inner loop, but after each PGD + eps-ball
   projection we additionally (a) snap `term_*` OHE columns to a valid
   one-hot, then (b) overwrite the `installment` column with the value
   derived from `(loan_amnt, int_rate, term)` via the standard loan
   amortisation formula, in raw space. Installment becomes a derived
   feature rather than an attack variable; its eps-ball is implicit
   through the other three.

**Produces**
- `results/adv_examples/g1_projection/lcld_neural_{unconstrained,g1proj}_seed{s}.parquet`
- `g1_projection_results.csv` - per-(seed, attack) metrics + feasibility + filtered success rate
- `g1_projection_summary.csv` - mean +/- std across seeds, constrained vs unconstrained
- `g1_projection_bars.png` - side-by-side comparison bar chart

**Scope.** LCLD only (the only dataset in the benchmark that carries a
nonlinear closed-form constraint). 3 seeds, eps=0.1, sample_frac=0.1,
identical model hyperparameters to mask_ablation and cross_dataset runs so
the constrained numbers are directly comparable.

**Bootstrap cells 1-5** mirror `notebooks/mask_ablation.ipynb` verbatim -
Drive mount, repo clone, deps install (requires runtime restart), dataset
symlinks.


In [ ]:
# Cell 1: Verify GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")


In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/FraudBench"
for subdir in ["data", "results", "results/adv_examples"]:
    os.makedirs(os.path.join(DRIVE_ROOT, subdir), exist_ok=True)
print("Google Drive mounted.")


In [ ]:
# Cell 3: Clone or update repo
import os, shutil

REPO_URL = "https://github.com/iHaydenzZ/Capstone_FraudBench.git"
REPO_DIR = "/content/Capstone_FraudBench"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    os.chdir(REPO_DIR)
    !git pull
else:
    os.chdir("/content")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f"Working directory: {os.getcwd()}")
!git log --oneline -3


In [ ]:
# Cell 4: Install dependencies
!pip install "numpy<2.1" "scipy>=1.14,<1.15" "scikit-learn>=1.5" -q 2>&1 | tail -5
!pip install -e . --no-deps -q 2>&1 | tail -5
!pip install "numba>=0.61" -q 2>&1 | tail -3
!pip install xgboost torch art pyyaml joblib pandas matplotlib -q 2>&1 | tail -3

# --- IMPORTANT ---
# After this cell finishes, restart the runtime:
#   Runtime > Restart session  (or Ctrl+M then .)
# Then skip this cell and continue from Cell 5.
print("\n>>> RESTART THE RUNTIME NOW, then skip this cell and run from Cell 5. <<<")


In [ ]:
# Cell 5: Symlink datasets from Google Drive
import os

DRIVE_DATA = "/content/drive/MyDrive/FraudBench/data"
DATASETS_DIR = "/content/Capstone_FraudBench/datasets"

for dataset_dir in ["CCFD", "ieee-fraud-detection", "LCLD", "Sparkov"]:
    src = os.path.join(DRIVE_DATA, dataset_dir)
    dst = os.path.join(DATASETS_DIR, dataset_dir)
    if os.path.islink(dst):
        os.unlink(dst)
    if os.path.exists(src):
        os.symlink(src, dst)
        print(f"  Linked: {dataset_dir}/")
    else:
        print(f"  NOT FOUND: {dataset_dir}/ -- upload to {src}")

print("Dataset symlinks ready.")


In [ ]:
# Cell 6: Experiment configuration
import os
import numpy as np
import pandas as pd

SEEDS = [42, 123, 456]
EPSILON = 0.1
SAMPLE_FRAC = 0.1
ATTACK_PARAMS = {"epsilon": EPSILON, "steps": 10, "step_size": EPSILON / 4}
MODEL_PARAMS = {"epochs": 20, "hidden_dim": 128, "batch_size": 256, "lr": 0.001}

G1_TOL = 0.10
TOL_OHE = 0.01

OUT_DIR = "results/adv_examples/g1_projection"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Output dir: {OUT_DIR}")
print(f"Plan: {len(SEEDS)} seeds x 2 attacks (unconstrained, g1-projected) = {len(SEEDS)*2} runs")


In [ ]:
# Cell 7: LCLD constraint checkers (g1-g4 + aggregate) and raw-space utilities
import re

def _to_float(series):
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(float)
    return pd.to_numeric(
        series.astype(str).str.replace(r"[^\d.\-]", "", regex=True),
        errors="coerce",
    )


def get_scaler_and_num_names(preprocessor):
    for name, transformer, columns in preprocessor.pipeline.transformers_:
        if name == "num":
            return transformer.named_steps["scaler"], list(columns)
    raise RuntimeError("No 'num' branch on the preprocessor")


def inverse_transform_numeric(X_proc, num_feature_names, scaler):
    sanitize = lambda c: re.sub(r"[\[\]<>]", "_", c)
    sanitized_num = [sanitize(c) for c in num_feature_names]
    proc_cols = X_proc.columns.tolist()
    matched = [(raw, san) for raw, san in zip(num_feature_names, sanitized_num)
               if san in proc_cols]
    raw_names = [m[0] for m in matched]
    san_names = [m[1] for m in matched]
    idx_in_scaler = [num_feature_names.index(r) for r in raw_names]
    X_scaled = X_proc[san_names].values
    means = scaler.mean_[idx_in_scaler]
    scales = scaler.scale_[idx_in_scaler]
    return pd.DataFrame(X_scaled * scales + means, columns=raw_names, index=X_proc.index)


def reconstruct_term_from_ohe(X_proc):
    term_cols = [c for c in X_proc.columns if c.startswith("term_")]
    if not term_cols:
        return None
    term_vals = {}
    for col in term_cols:
        v = pd.to_numeric(col.replace("term_", "").replace("months", "").strip(),
                          errors="coerce")
        if not np.isnan(v):
            term_vals[col] = v
    if not term_vals:
        return None
    term_df = X_proc[list(term_vals.keys())]
    return term_df.idxmax(axis=1).map(term_vals)


def check_g1_installment(df, tol=G1_TOL):
    loan = _to_float(df["loan_amnt"])
    rate = _to_float(df["int_rate"])
    term = _to_float(df["term"])
    inst = _to_float(df["installment"])
    r = rate / 12.0 / 100.0
    with np.errstate(divide="ignore", invalid="ignore"):
        expected = loan * r * (1 + r) ** term / ((1 + r) ** term - 1)
    return (inst - expected).abs() < tol


def check_g2_open_total(df):
    return _to_float(df["open_acc"]) <= _to_float(df["total_acc"])


def check_g3_bankruptcies(df):
    return _to_float(df["pub_rec_bankruptcies"]) <= _to_float(df["pub_rec"])


def check_g4_term_ohe(X_proc, tol=TOL_OHE):
    cols = [c for c in X_proc.columns if c.startswith("term_")]
    if not cols:
        return None
    ohe = X_proc[cols].values
    valid = (np.abs(ohe.sum(axis=1) - 1.0) < tol) & (np.abs(ohe.max(axis=1) - 1.0) < tol)
    return pd.Series(valid, index=X_proc.index)


def _pass(s):
    return float(s.fillna(True).mean()) if s is not None else np.nan


def lcld_feasibility(X_raw, X_proc):
    X_raw = X_raw.copy()
    t = reconstruct_term_from_ohe(X_proc)
    if t is not None:
        X_raw["term"] = t.values
    g1 = check_g1_installment(X_raw)
    g2 = check_g2_open_total(X_raw)
    g3 = check_g3_bankruptcies(X_raw)
    g4 = check_g4_term_ohe(X_proc)
    per = {
        "g1_installment": _pass(g1),
        "g2_open_total":  _pass(g2),
        "g3_bankruptcy":  _pass(g3),
        "g4_term_ohe":    _pass(g4),
    }
    idx = X_raw.index
    def _b(s): return s.fillna(True).astype(bool) if s is not None else pd.Series(True, index=idx)
    agg = float((_b(g1) & _b(g2) & _b(g3) & _b(g4)).mean())
    return per, agg


In [ ]:
# Cell 8: Projection metadata - build once per fitted preprocessor
# g1 projection needs scaler stats for loan_amnt, int_rate, installment and
# the processed-column indices of all three. Term projection needs the
# indices of all term_* OHE columns together with their raw numeric values
# (parsed from the column name, e.g. 'term_ 36 months' -> 36).

def build_term_proj_info(processed_feature_names):
    indices, values = [], []
    for i, col in enumerate(processed_feature_names):
        if not col.startswith("term_"):
            continue
        v = pd.to_numeric(col.replace("term_", "").replace("months", "").strip(),
                          errors="coerce")
        if np.isnan(v):
            continue
        indices.append(i)
        values.append(float(v))
    if not indices:
        raise RuntimeError("No term_ OHE columns found in processed feature space")
    return {"indices": indices, "values": values}


def build_g1_proj_info(processed_feature_names, scaler, num_feature_names):
    def _locate(raw_name):
        proc_idx = processed_feature_names.index(raw_name)
        s_idx = num_feature_names.index(raw_name)
        return proc_idx, float(scaler.mean_[s_idx]), float(scaler.scale_[s_idx])
    idx_loan, m_loan, s_loan = _locate("loan_amnt")
    idx_rate, m_rate, s_rate = _locate("int_rate")
    idx_inst, m_inst, s_inst = _locate("installment")
    return {
        "idx_loan": idx_loan, "mean_loan": m_loan, "scale_loan": s_loan,
        "idx_rate": idx_rate, "mean_rate": m_rate, "scale_rate": s_rate,
        "idx_inst": idx_inst, "mean_inst": m_inst, "scale_inst": s_inst,
    }


def project_term_ohe_tensor(x_adv, term_info):
    '''Snap term_* OHE columns to a valid one-hot (argmax -> 1, rest -> 0).
    In place on x_adv (which is expected to be inside a torch.no_grad block).'''
    idx = term_info["indices"]
    ohe = x_adv[:, idx]
    argmax = ohe.argmax(dim=1)
    snapped = torch.zeros_like(ohe)
    snapped[torch.arange(ohe.size(0), device=x_adv.device), argmax] = 1.0
    x_adv[:, idx] = snapped
    return x_adv


def project_g1_tensor(x_adv, g1_info, term_info):
    '''Overwrite x_adv's installment column with the value derived from
    loan_amnt, int_rate, term via the standard amortisation formula.
    Operates in processed (scaled) space by inverse-transforming the three
    inputs, computing in raw space, then forward-transforming the output.
    Call project_term_ohe_tensor first so term argmax is meaningful.'''
    loan_proc = x_adv[:, g1_info["idx_loan"]]
    rate_proc = x_adv[:, g1_info["idx_rate"]]
    loan_raw = loan_proc * g1_info["scale_loan"] + g1_info["mean_loan"]
    rate_raw = rate_proc * g1_info["scale_rate"] + g1_info["mean_rate"]

    term_ohe = x_adv[:, term_info["indices"]]
    argmax = term_ohe.argmax(dim=1)
    term_values = torch.as_tensor(term_info["values"],
                                   device=x_adv.device, dtype=x_adv.dtype)
    term_raw = term_values[argmax]

    r = rate_raw / 12.0 / 100.0
    factor = (1.0 + r) ** term_raw
    expected_raw = loan_raw * r * factor / (factor - 1.0 + 1e-12)
    expected_proc = (expected_raw - g1_info["mean_inst"]) / g1_info["scale_inst"]
    x_adv[:, g1_info["idx_inst"]] = expected_proc
    return x_adv


In [ ]:
# Cell 9: Constrained CAPGD - copy of attacks.capgd.capgd_attack with per-step
# g1 + term-OHE projection inserted after the schema projection.
import torch
import torch.nn as nn
from typing import Dict, Any
from attacks.capgd import project_constraints
from constraints.schema import ConstraintSchema


def capgd_attack_g1_projected(
    model,
    X: pd.DataFrame,
    y: pd.Series,
    schema: ConstraintSchema,
    feature_types: Dict[str, str],
    g1_info: dict,
    term_info: dict,
    params: Dict[str, Any] = None,
) -> pd.DataFrame:
    if params is None:
        params = {}
    epsilon = params.get("epsilon", 0.1)
    steps = params.get("steps", 10)
    step_size = params.get("step_size", epsilon / 4)

    if not hasattr(model, "model") or not isinstance(model.model, nn.Module):
        print("Warning: non-PyTorch model; returning clean data.")
        return X

    torch_model = model.model
    device = model.device
    torch_model.train(False)  # inference mode (equivalent to .eval())

    X_tensor = torch.tensor(X.values, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1).to(device)
    feature_names = X.columns.tolist()

    noise = torch.zeros_like(X_tensor).uniform_(-epsilon, epsilon)
    x_adv = X_tensor + noise
    x_adv = project_constraints(x_adv, X_tensor, schema, feature_names, feature_types)
    with torch.no_grad():
        x_adv = project_term_ohe_tensor(x_adv, term_info)
        x_adv = project_g1_tensor(x_adv, g1_info, term_info)
    x_adv = x_adv.detach()
    x_adv.requires_grad = True

    use_logits = hasattr(model, "_use_logits") and model._use_logits
    criterion = nn.BCEWithLogitsLoss() if use_logits else nn.BCELoss()

    for _ in range(steps):
        outputs = torch_model(x_adv)
        loss = criterion(outputs, y_tensor)
        torch_model.zero_grad()
        loss.backward()

        with torch.no_grad():
            grad = x_adv.grad
            x_adv = x_adv + step_size * grad.sign()
            if epsilon > 0:
                delta = torch.clamp(x_adv - X_tensor, -epsilon, epsilon)
                x_adv = X_tensor + delta
            x_adv = project_constraints(x_adv, X_tensor, schema, feature_names, feature_types)
            x_adv = project_term_ohe_tensor(x_adv, term_info)
            x_adv = project_g1_tensor(x_adv, g1_info, term_info)
            x_adv.requires_grad = True

    return pd.DataFrame(x_adv.detach().cpu().numpy(), columns=feature_names, index=X.index)


In [ ]:
# Cell 10: Main loop - train once per seed, run both attacks
import time
from datasets.loader import load_dataset
from datasets.splitter import split_dataset
from preprocessing.processor import DataPreprocessor
from constraints.schema import ConstraintSchema
from models.neural import NeuralModel
from evaluation.metrics import compute_metrics
from attacks.capgd import capgd_attack

rows = []

for seed in SEEDS:
    print(f"\n{'='*60}\n  SEED = {seed}\n{'='*60}")

    dataset = load_dataset("lcld", config={"sample_frac": SAMPLE_FRAC})
    X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(
        dataset, test_size=0.2, val_size=0.2, random_state=seed,
    )
    preprocessor = DataPreprocessor(dataset.feature_types)
    X_train_p = preprocessor.fit_transform(X_train)
    X_test_p  = preprocessor.transform(X_test)
    proc_ft = {c: "numeric" for c in X_train_p.columns}
    schema = ConstraintSchema.from_data(X_train_p, proc_ft)

    scaler, num_names = get_scaler_and_num_names(preprocessor)
    term_info = build_term_proj_info(X_train_p.columns.tolist())
    g1_info   = build_g1_proj_info(X_train_p.columns.tolist(), scaler, num_names)
    print(f"  proc_dim={X_test_p.shape[1]}  term_cols={len(term_info['indices'])}  "
          f"idx_inst={g1_info['idx_inst']}")

    model = NeuralModel(MODEL_PARAMS)
    t0 = time.time()
    model.fit(X_train_p, y_train)
    print(f"  trained in {time.time()-t0:.1f}s")
    clean_probs = model.predict_proba(X_test_p)
    clean_m = compute_metrics(y_test, clean_probs)
    print(f"  clean  PR-AUC={clean_m['pr_auc']:.4f}  Acc={clean_m['accuracy']:.4f}  "
          f"Rec={clean_m.get('recall', float('nan')):.4f}")

    X_test_raw = inverse_transform_numeric(X_test_p, num_names, scaler)
    _, clean_agg = lcld_feasibility(X_test_raw, X_test_p)

    # Unconstrained CAPGD
    t0 = time.time()
    X_adv_u = capgd_attack(model, X_test_p, y_test, schema, proc_ft, params=ATTACK_PARAMS)
    dt_u = time.time() - t0
    probs_u = model.predict_proba(X_adv_u)
    m_u = compute_metrics(y_test, probs_u)
    X_adv_u_raw = inverse_transform_numeric(X_adv_u, num_names, scaler)
    per_u, agg_u = lcld_feasibility(X_adv_u_raw, X_adv_u)

    # g1-projected CAPGD
    t0 = time.time()
    X_adv_p = capgd_attack_g1_projected(
        model, X_test_p, y_test, schema, proc_ft,
        g1_info, term_info, params=ATTACK_PARAMS,
    )
    dt_p = time.time() - t0
    probs_p = model.predict_proba(X_adv_p)
    m_p = compute_metrics(y_test, probs_p)
    X_adv_p_raw = inverse_transform_numeric(X_adv_p, num_names, scaler)
    per_p, agg_p = lcld_feasibility(X_adv_p_raw, X_adv_p)

    parq_u = os.path.join(OUT_DIR, f"lcld_neural_unconstrained_seed{seed}.parquet")
    parq_p = os.path.join(OUT_DIR, f"lcld_neural_g1proj_seed{seed}.parquet")
    X_adv_u.to_parquet(parq_u)
    X_adv_p.to_parquet(parq_p)

    clean_pred = (clean_probs >= 0.5).astype(int)
    u_pred     = (probs_u    >= 0.5).astype(int)
    p_pred     = (probs_p    >= 0.5).astype(int)
    pos_mask = (y_test.values == 1)
    flipped_u = int(((clean_pred == 1) & (u_pred == 0) & pos_mask).sum())
    flipped_p = int(((clean_pred == 1) & (p_pred == 0) & pos_mask).sum())

    def _agg_mask(X_raw, X_p):
        t = reconstruct_term_from_ohe(X_p); X_raw = X_raw.copy()
        if t is not None: X_raw["term"] = t.values
        g1 = check_g1_installment(X_raw).fillna(True).astype(bool)
        g2 = check_g2_open_total(X_raw).fillna(True).astype(bool)
        g3 = check_g3_bankruptcies(X_raw).fillna(True).astype(bool)
        g4 = check_g4_term_ohe(X_p)
        g4 = g4.fillna(True).astype(bool) if g4 is not None else pd.Series(True, index=X_raw.index)
        return (g1 & g2 & g3 & g4).values

    mask_u = _agg_mask(X_adv_u_raw, X_adv_u)
    mask_p = _agg_mask(X_adv_p_raw, X_adv_p)
    flipped_feas_u = int(((clean_pred == 1) & (u_pred == 0) & pos_mask & mask_u).sum())
    flipped_feas_p = int(((clean_pred == 1) & (p_pred == 0) & pos_mask & mask_p).sum())

    print(f"  unconst  PR-AUC={m_u['pr_auc']:.4f}  Acc={m_u['accuracy']:.4f}  "
          f"agg_feas={agg_u:.4f}  g1={per_u['g1_installment']:.4f}  "
          f"flipped={flipped_u} feas-flip={flipped_feas_u}  ({dt_u:.1f}s)")
    print(f"  g1proj   PR-AUC={m_p['pr_auc']:.4f}  Acc={m_p['accuracy']:.4f}  "
          f"agg_feas={agg_p:.4f}  g1={per_p['g1_installment']:.4f}  "
          f"flipped={flipped_p} feas-flip={flipped_feas_p}  ({dt_p:.1f}s)")

    def _row(attack, metrics, per, agg, flipped, flipped_feas, dt):
        r = {
            "seed": seed, "attack": attack,
            "clean_pr_auc": clean_m["pr_auc"], "clean_accuracy": clean_m["accuracy"],
            "clean_recall": clean_m.get("recall", np.nan),
            "robust_pr_auc": metrics["pr_auc"], "robust_accuracy": metrics["accuracy"],
            "robust_recall": metrics.get("recall", np.nan),
            "clean_feasibility": clean_agg,
            "adv_feasibility":   agg,
            "flipped_positives": flipped,
            "feasible_flipped":  flipped_feas,
            "attack_time_sec":   round(dt, 1),
        }
        for k, v in per.items(): r[f"adv_{k}"] = v
        return r

    rows.append(_row("unconstrained", m_u, per_u, agg_u, flipped_u, flipped_feas_u, dt_u))
    rows.append(_row("g1proj",        m_p, per_p, agg_p, flipped_p, flipped_feas_p, dt_p))

results_df = pd.DataFrame(rows)
results_csv = os.path.join(OUT_DIR, "g1_projection_results.csv")
results_df.to_csv(results_csv, index=False)
print(f"\nSaved {len(results_df)} rows -> {results_csv}")
results_df


In [ ]:
# Cell 11: Summary - mean +/- std across seeds, constrained vs unconstrained
agg_cols = {
    "robust_pr_auc":     ["mean", "std"],
    "robust_accuracy":   ["mean", "std"],
    "robust_recall":     ["mean", "std"],
    "adv_feasibility":   ["mean", "std"],
    "adv_g1_installment":["mean", "std"],
    "adv_g2_open_total": ["mean", "std"],
    "adv_g3_bankruptcy": ["mean", "std"],
    "adv_g4_term_ohe":   ["mean", "std"],
    "flipped_positives": ["mean"],
    "feasible_flipped":  ["mean"],
}
summary = results_df.groupby("attack").agg(agg_cols)
summary.columns = ["_".join(c).rstrip("_") for c in summary.columns]
summary = summary.reset_index()

summary["filtered_success_rate"] = (
    summary["feasible_flipped_mean"] / summary["flipped_positives_mean"].replace(0, np.nan)
)

summary_csv = os.path.join(OUT_DIR, "g1_projection_summary.csv")
summary.to_csv(summary_csv, index=False)

print("Constrained vs unconstrained (mean +/- std over 3 seeds):")
cols_display = [
    "attack",
    "robust_pr_auc_mean", "robust_pr_auc_std",
    "robust_accuracy_mean", "robust_accuracy_std",
    "adv_feasibility_mean", "adv_feasibility_std",
    "adv_g1_installment_mean",
    "flipped_positives_mean", "feasible_flipped_mean",
    "filtered_success_rate",
]
print(summary[cols_display].to_string(index=False))


In [ ]:
# Cell 12: Drive backup + comparison plot
import shutil
import matplotlib.pyplot as plt

DRIVE_OUT = "/content/drive/MyDrive/FraudBench/results/adv_examples/g1_projection"
os.makedirs(DRIVE_OUT, exist_ok=True)
for f in sorted(os.listdir(OUT_DIR)):
    shutil.copy(os.path.join(OUT_DIR, f), os.path.join(DRIVE_OUT, f))
print(f"Backed up {len(os.listdir(OUT_DIR))} files -> {DRIVE_OUT}")

metrics = [
    ("robust_pr_auc",     "Robust PR-AUC"),
    ("robust_accuracy",   "Robust accuracy"),
    ("adv_g1_installment","g1 pass rate"),
    ("adv_feasibility",   "Aggregate feasibility"),
]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, (metric, label) in zip(axes, metrics):
    sub = summary.set_index("attack")
    means = sub[f"{metric}_mean"]
    stds  = sub.get(f"{metric}_std", pd.Series(0, index=sub.index)).fillna(0)
    x = np.arange(len(means))
    ax.bar(x, means.values, yerr=stds.values, capsize=4,
           color=["#888", "#2a7"][:len(means)])
    ax.set_xticks(x)
    ax.set_xticklabels(means.index)
    ax.set_title(label)
    for i, v in enumerate(means.values):
        ax.text(i, v + 0.01 * max(means.values.max(), 0.01),
                f"{v:.3f}", ha="center", fontsize=9)
plt.suptitle("g1-projected vs unconstrained CAPGD (LCLD, 3 seeds)")
plt.tight_layout()
fig_path = os.path.join(OUT_DIR, "g1_projection_bars.png")
plt.savefig(fig_path, dpi=150)
shutil.copy(fig_path, os.path.join(DRIVE_OUT, "g1_projection_bars.png"))
plt.show()
print(f"Saved comparison plot -> {fig_path}")
